# 01 — Instrument Tracker: Full Training Pipeline\n\n---\n\n## What This Notebook Does\n\nThis notebook walks you through the **entire** Instrument Tracker pipeline — from pulling data out of the SurgicalInstruments catalog app, all the way to a trained YOLOv8 detector ready for production.\n\n**You don't need to write any Python.** Every step calls an existing script. The notebook is here to:\n1. Guide you through the process in order\n2. Show you what each step produces\n3. Let you inspect results between steps\n4. Document what to look for at each stage\n\n---\n\n## Prerequisites\n\n| Requirement | How to verify |\n|---|---|\n| **SurgicalInstruments app running** | Open http://localhost:3000 — you should see the dashboard |\n| **Conda env `sam3` active** | This notebook should run inside the `sam3` environment |\n| **GPU available** | Steps 5+ need a CUDA GPU for training |\n| **Catalog has data** | The app must have kits + instruments with images uploaded |\n\n---\n\n## Pipeline Overview\n\n```\nSurgicalInstruments App (port 3000)\n        │\n  STEP 1: Sync catalog data (taxonomy, kits, images, videos)\n        │\n  STEP 2: Inspect & validate what came through\n        │\n  STEP 3: Deduplicate instrument names across kits\n        │\n  STEP 4: Build YOLO dataset (images + labels + splits)\n        │\n  STEP 5: Train YOLOv8-seg detector\n        │\n  STEP 6: Evaluate detection performance\n        │\n  STEP 7: (Future) Add OR video data & retrain\n```

---\n\n## Setup (run once)\n\nThis cell sets paths and checks that the catalog app is reachable. If the health check fails, start the SurgicalInstruments app first:\n\n```\ncd SurgicalInstruments && node server.js\n```

In [ ]:
import os, json, subprocess, urllib.request\nfrom pathlib import Path\n\n# ── Paths ──\nCOMPONENT = Path.cwd()  # should be 01_instrument_tracker/\nDATA      = COMPONENT / \"data\"\nCONFIG    = COMPONENT / \"config\" / \"config.yaml\"\nAPI_URL   = \"http://localhost:3000\"\n\n# ── Health check ──\ntry:\n    resp = json.loads(urllib.request.urlopen(f\"{API_URL}/api/health\", timeout=5).read())\n    print(f\"✓ Catalog app is UP  —  {resp.get('instruments', '?')} instruments, {resp.get('images', '?')} images\")\nexcept Exception as e:\n    print(f\"✗ Cannot reach {API_URL} — start the SurgicalInstruments app first.\\n  Error: {e}\")

---\n\n# STEP 1 — Sync from Catalog\n\nPulls everything we need from the SurgicalInstruments app:\n\n| What | Where it goes | Used for |\n|---|---|---|\n| Instrument names + families | `data/taxonomy.json` | Training class labels |\n| Kit definitions | `data/kits/` | Count validation at inference |\n| Instrument images (stills) | `data/exemplars/` | Training images |\n| Instrument videos (clips) | `data/exemplars/` | Frame extraction → more training images |\n| Kit instrument counts | `data/kit_instrument_counts.json` | Ground truth for count alerts |\n\n**Why this matters:** The catalog is the single source of truth. Every instrument name, every image, every kit definition comes from here. When someone adds a new instrument to the catalog, you re-run this cell and it appears in training.

In [ ]:
## ── 1A: Pull taxonomy + kit definitions ──\n\nresp = json.loads(urllib.request.urlopen(f\"{API_URL}/api/instruments\").read())\nkits = json.loads(urllib.request.urlopen(f\"{API_URL}/api/kit-catalogs\").read())\nfamilies = json.loads(urllib.request.urlopen(f\"{API_URL}/api/families\").read())\n\n# Save locally\nDATA.mkdir(parents=True, exist_ok=True)\n(DATA / \"kits\").mkdir(exist_ok=True)\n(DATA / \"exemplars\").mkdir(exist_ok=True)\n\nwith open(DATA / \"taxonomy.json\", \"w\") as f:\n    json.dump(resp, f, indent=2)\nwith open(DATA / \"families.json\", \"w\") as f:\n    json.dump(families, f, indent=2)\nfor kit in kits:\n    with open(DATA / \"kits\" / f\"kit_{kit.get('id','unknown')}.json\", \"w\") as f:\n        json.dump(kit, f, indent=2)\n\nprint(f\"{'─'*50}\")\nprint(f\"  STEP 1A RESULTS\")\nprint(f\"{'─'*50}\")\nprint(f\"  Instruments pulled:  {len(resp)}\")\nprint(f\"  Kit catalogs:        {len(kits)}\")\nprint(f\"  Families:            {len(families)}\")\nprint(f\"{'─'*50}\")\nprint(f\"\\n  Sample instruments (first 5):\")\nfor inst in resp[:5]:\n    imgs = inst.get('image_count', inst.get('imageCount', '?'))\n    print(f\"    • {inst.get('name','?'):40s}  family={inst.get('family','?'):20s}  images={imgs}\")\nif len(resp) > 5:\n    print(f\"    ... and {len(resp)-5} more\")\nprint(f\"\\n  Sample kits:\")\nfor kit in kits[:3]:\n    print(f\"    • {kit.get('name','?'):40s}  instruments={kit.get('instrumentCount', len(kit.get('instruments',[])))}\") 

### 1B — Incremental Download: Images & Video Frames\n\nThis step uses a **sync manifest** (`data/sync_manifest.json`) to track what's already been downloaded. On each run it:\n\n1. Compares the catalog's `updatedAt` timestamp and `imageCount` per instrument against the manifest\n2. **NEW** instrument (not in manifest) → downloads all images/videos\n3. **UPDATED** instrument (imageCount changed or newer updatedAt) → re-downloads\n4. **UNCHANGED** → skips entirely (fast!)\n\nFor video files, it extracts 10 random frames as JPEGs for training diversity.\n\nThe API returns **binary data** (not JSON) — images are saved directly to disk.\n\nAfter download, a **verification step** confirms that files are valid images/videos (not corrupt or empty).

In [ ]:
## ── 1B: Incremental download with sync manifest ──
import re, sys, random, cv2
from datetime import datetime

def safe_dirname(name):
    """Sanitize instrument name for use as folder name."""
    return re.sub(r'[^\w\s-]', '', name).strip().replace(' ', '_')[:80]

def progress_bar(current, total, prefix='', width=40):
    pct = current / max(total, 1)
    filled = int(width * pct)
    bar = '█' * filled + '░' * (width - filled)
    sys.stdout.write(f'\r  {prefix} [{bar}] {current}/{total} ({pct:.0%})')
    sys.stdout.flush()

VIDEO_EXTS = {'.mp4', '.webm', '.mov', '.avi', '.mkv'}
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
FRAMES_PER_VIDEO = 10

# ── Load existing manifest (if any) ──
MANIFEST_PATH = DATA / "sync_manifest.json"
if MANIFEST_PATH.exists():
    with open(MANIFEST_PATH) as f:
        manifest = json.load(f)
    print(f"  Loaded existing manifest from {manifest.get('last_sync', '?')}")
    print(f"  Previously synced: {len(manifest.get('instruments', {}))} instruments")
else:
    manifest = {"instruments": {}}
    print("  No previous manifest — full download will run.")

prev = manifest.get("instruments", {})

# ── Determine what needs downloading ──
to_download = []  # (inst, reason)
to_skip = []

for inst in resp:
    iid = str(inst['id'])
    name = inst.get('name', '?')
    img_count = inst.get('imageCount', inst.get('image_count', 0))
    updated = inst.get('updatedAt', inst.get('updated_at', ''))
    
    if iid not in prev:
        to_download.append((inst, "NEW"))
    elif prev[iid].get('imageCount', 0) != img_count:
        to_download.append((inst, f"UPDATED (images {prev[iid].get('imageCount',0)}→{img_count})"))
    elif prev[iid].get('updatedAt', '') != updated:
        to_download.append((inst, "UPDATED (modified)"))
    else:
        to_skip.append(inst)

print(f"\n{'─'*50}")
print(f"  SYNC PLAN")
print(f"{'─'*50}")
print(f"  To download:   {len(to_download):>4d}  instruments")
print(f"  Skipped:       {len(to_skip):>4d}  (unchanged)")
print(f"{'─'*50}")
if to_download:
    print(f"\n  Downloading:")
    for inst, reason in to_download[:10]:
        print(f"    • {inst.get('name','?'):40s}  [{reason}]")
    if len(to_download) > 10:
        print(f"    ... and {len(to_download)-10} more")

# ── Download loop ──
total_images = 0
total_videos = 0
total_frames = 0
skipped_files = 0
errors = []
new_manifest = manifest.get("instruments", {}).copy()

for idx, (inst, reason) in enumerate(to_download):
    progress_bar(idx + 1, len(to_download), prefix='Downloading')
    
    iid = str(inst['id'])
    inst_name = inst.get('name', f"unknown_{iid}")
    inst_dir = DATA / "exemplars" / safe_dirname(inst_name)
    inst_dir.mkdir(parents=True, exist_ok=True)
    
    # Fetch full instrument detail (includes images list)
    try:
        detail = json.loads(urllib.request.urlopen(f"{API_URL}/api/instruments/{iid}").read())
    except Exception as e:
        errors.append(f"{inst_name}: {e}")
        continue
    
    inst_files = []
    for img in detail.get('images', []):
        filename = img.get('filename', '')
        if not filename:
            continue
        out_path = inst_dir / filename
        
        try:
            # API returns BINARY data, not JSON
            req = urllib.request.Request(f"{API_URL}/api/images/{filename}/data")
            with urllib.request.urlopen(req, timeout=30) as response:
                raw = response.read()
            
            if len(raw) < 100:  # too small = probably error
                skipped_files += 1
                continue
            
            out_path.write_bytes(raw)
            ext = Path(filename).suffix.lower()
            
            if ext in VIDEO_EXTS:
                total_videos += 1
                inst_files.append({"filename": filename, "type": "video", "size": len(raw)})
                # Extract frames from video
                cap = cv2.VideoCapture(str(out_path))
                frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                if frame_count > 0:
                    n_frames = min(FRAMES_PER_VIDEO, frame_count)
                    indices = sorted(random.sample(range(frame_count), n_frames))
                    for fi, frame_idx in enumerate(indices):
                        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                        ret, frame = cap.read()
                        if ret:
                            frame_path = inst_dir / f"{Path(filename).stem}_frame{fi:03d}.jpg"
                            cv2.imwrite(str(frame_path), frame)
                            total_frames += 1
                cap.release()
            else:
                total_images += 1
                inst_files.append({"filename": filename, "type": "image", "size": len(raw)})
                
        except Exception as e:
            errors.append(f"{inst_name}/{filename}: {e}")
    
    # Update manifest entry
    new_manifest[iid] = {
        "name": inst_name,
        "imageCount": inst.get('imageCount', inst.get('image_count', 0)),
        "videoCount": inst.get('videoCount', inst.get('video_count', 0)),
        "updatedAt": inst.get('updatedAt', inst.get('updated_at', '')),
        "files": inst_files,
        "synced_at": datetime.now().isoformat()
    }

# ── Save updated manifest ──
manifest_out = {
    "last_sync": datetime.now().isoformat(),
    "api_url": API_URL,
    "total_instruments": len(resp),
    "instruments": new_manifest
}
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest_out, f, indent=2)

# ── Results ──
if to_download:
    print(f"\n")
print(f"\n{'─'*50}")
print(f"  STEP 1B RESULTS")
print(f"{'─'*50}")
print(f"  Images downloaded:       {total_images}")
print(f"  Videos downloaded:       {total_videos}")
print(f"  Frames extracted:        {total_frames}")
print(f"  Skipped (unchanged):     {len(to_skip)} instruments")
print(f"  Skipped (bad data):      {skipped_files} files")
print(f"  Errors:                  {len(errors)}")
print(f"  Total NEW training imgs: {total_images + total_frames}")
print(f"  Manifest saved to:       {MANIFEST_PATH}")
print(f"{'─'*50}")

# ── Show sample downloads ──
exemplar_dirs = sorted(DATA.glob("exemplars/*/"))
print(f"\n  Sample downloads (first 5 instruments):")
for d in exemplar_dirs[:5]:
    files = list(d.glob("*"))
    imgs = [f for f in files if f.suffix.lower() in IMAGE_EXTS]
    vids = [f for f in files if f.suffix.lower() in VIDEO_EXTS]
    print(f"    • {d.name:40s}  {len(imgs)} images, {len(vids)} videos")

if errors:
    print(f"\n  ⚠ Errors (first 5):")
    for e in errors[:5]:
        print(f"    {e}")

### 1C — Verify Downloaded Files\n\nThis step checks that every downloaded file is actually a valid image or video — not a corrupt file, an HTML error page, or an empty response. It also shows sample thumbnails so you can visually confirm the data looks right.

In [ ]:
## ── 1C: Verify all downloaded files are valid ──
import matplotlib.pyplot as plt

valid_images = []
valid_videos = []
corrupt_files = []
empty_dirs = []

exemplar_dirs = sorted(DATA.glob("exemplars/*/"))
for d_idx, d in enumerate(exemplar_dirs):
    progress_bar(d_idx + 1, len(exemplar_dirs), prefix='Verifying')
    files = list(d.iterdir())
    
    if not files:
        empty_dirs.append(d.name)
        continue
    
    for f in files:
        ext = f.suffix.lower()
        if ext in IMAGE_EXTS:
            img = cv2.imread(str(f))
            if img is not None and img.shape[0] > 10 and img.shape[1] > 10:
                valid_images.append(f)
            else:
                corrupt_files.append(f)
        elif ext in VIDEO_EXTS:
            cap = cv2.VideoCapture(str(f))
            if cap.isOpened() and cap.get(cv2.CAP_PROP_FRAME_COUNT) > 0:
                valid_videos.append(f)
            else:
                corrupt_files.append(f)
            cap.release()

print(f"\n\n{'─'*50}")
print(f"  STEP 1C: VERIFICATION")
print(f"{'─'*50}")
print(f"  Valid images:     {len(valid_images)}")
print(f"  Valid videos:     {len(valid_videos)}")
print(f"  Corrupt/bad:      {len(corrupt_files)}")
print(f"  Empty folders:    {len(empty_dirs)}")
print(f"{'─'*50}")

if corrupt_files:
    print(f"\n  ⚠ Corrupt files (will be excluded from training):")
    for f in corrupt_files[:10]:
        print(f"    {f.parent.name}/{f.name}  ({f.stat().st_size} bytes)")
    # Delete corrupt files
    for f in corrupt_files:
        f.unlink()
    print(f"  → Deleted {len(corrupt_files)} corrupt files.")

if empty_dirs:
    print(f"\n  ⚠ Instrument folders with no data ({len(empty_dirs)}):")
    for name in empty_dirs[:10]:
        print(f"    • {name}")

# ── Show sample thumbnails: 3 random images + 1 video frame ──
print(f"\n  ── Sample thumbnails (verify visually) ──")
n_samples = min(4, len(valid_images))
if n_samples > 0:
    samples = random.sample(valid_images, n_samples)
    fig, axes = plt.subplots(1, n_samples, figsize=(4 * n_samples, 4))
    if n_samples == 1:
        axes = [axes]
    for ax, img_path in zip(axes, samples):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{img_path.parent.name}\n{img.shape[1]}x{img.shape[0]}", fontsize=8)
        ax.axis('off')
    plt.suptitle("Sample Downloaded Images", fontsize=12)
    plt.tight_layout()
    plt.show()

# Show a video frame sample if we have videos
if valid_videos:
    vid = random.choice(valid_videos)
    cap = cv2.VideoCapture(str(vid))
    ret, frame = cap.read()
    cap.release()
    if ret:
        fig, ax = plt.subplots(1, 1, figsize=(6, 4))
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Video frame: {vid.parent.name}/{vid.name}\n{frame.shape[1]}x{frame.shape[0]}", fontsize=9)
        ax.axis('off')
        plt.suptitle("Sample Video Frame", fontsize=12)
        plt.tight_layout()
        plt.show()
    print(f"\n  Video sample: {vid.parent.name}/{vid.name}")
    cap2 = cv2.VideoCapture(str(vid))
    fps = cap2.get(cv2.CAP_PROP_FPS)
    fc = int(cap2.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap2.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap2.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap2.release()
    print(f"    Resolution: {w}x{h}, FPS: {fps:.1f}, Frames: {fc}, Duration: {fc/max(fps,1):.1f}s")

---\n\n# STEP 2 — Deduplicate Instrument Names\n\nThe same physical instrument often appears with **different names** across kits:\n\n| Kit A | Kit B | Kit C |\n|---|---|---|\n| Richardson Retractor | Retractor, Richardson, Loop Handle, 38mm | Richardson retractor - large |\n\nAll three are the **same instrument** and must become **one training class**. If we don't merge them, the model wastes data learning 3 classes for 1 object.\n\n### Rules:\n- **Different COLOR** (blue vs gold handle) → **SAME class** (color doesn't matter functionally)\n- **Different SIZE** (14cm vs 20cm) → **DIFFERENT class** (nurse must distinguish)\n- **Different SHAPE** (curved vs straight) → **DIFFERENT class** (different instruments)\n\n### What this step does:\n1. Normalizes all names (lowercase, remove punctuation, remove filler words)\n2. Removes color words (blue, gold, silver, titanium...)\n3. Keeps size words (14cm, 20cm, large, small...)\n4. Groups by normalized key\n5. Saves `dedup_log.json` so you can review what was merged

In [ ]:
## ── 2: Deduplicate instrument names ──\n\nCOLOR_WORDS = {'blue','gold','black','silver','titanium','green','red','yellow',\n               'purple','white','chrome','stainless','steel','colored','coated'}\nFILLER_WORDS = {'the','a','an','with','for','and','or','of','in','on','type',\n                'style','handle','model','mm','instrument','surgical'}\n\ndef normalize_name(name):\n    \"\"\"Normalize instrument name: lowercase, remove colors + filler, sort tokens.\"\"\"\n    name = name.lower()\n    name = re.sub(r'[^a-z0-9\\s]', ' ', name)          # remove punctuation\n    tokens = name.split()\n    tokens = [t for t in tokens if t not in FILLER_WORDS and t not in COLOR_WORDS]\n    tokens = [t for t in tokens if len(t) > 1]          # remove single chars\n    return ' '.join(sorted(tokens))\n\n# Build dedup map\nname_groups = {}  # normalized_key → [original_names]\nname_to_dirs = {} # normalized_key → [exemplar_dirs]\n\nfor inst in resp:\n    orig_name = inst.get('name', '')\n    norm = normalize_name(orig_name)\n    if not norm:\n        continue\n    name_groups.setdefault(norm, []).append(orig_name)\n    dirname = safe_dirname(orig_name)\n    name_to_dirs.setdefault(norm, []).append(dirname)\n\n# Find actual duplicates (keys with multiple original names)\nmerged = {k: v for k, v in name_groups.items() if len(set(v)) > 1}\n\ndedup_log = {\n    \"total_instruments\": len(resp),\n    \"unique_classes\": len(name_groups),\n    \"merged_groups\": len(merged),\n    \"merges\": {k: list(set(v)) for k, v in merged.items()}\n}\n\nwith open(DATA / \"dedup_log.json\", \"w\") as f:\n    json.dump(dedup_log, f, indent=2)\n\n# Build class map\nclass_map = {}\nfor idx, norm_key in enumerate(sorted(name_groups.keys())):\n    representative = name_groups[norm_key][0]  # first seen name\n    class_map[str(idx)] = {\n        \"name\": representative,\n        \"normalized\": norm_key,\n        \"original_names\": list(set(name_groups[norm_key])),\n        \"exemplar_dirs\": list(set(name_to_dirs[norm_key]))\n    }\n\nwith open(DATA / \"class_map.json\", \"w\") as f:\n    json.dump(class_map, f, indent=2)\n\nprint(f\"{'─'*50}\")\nprint(f\"  STEP 2 RESULTS\")\nprint(f\"{'─'*50}\")\nprint(f\"  Original instrument names:  {len(resp)}\")\nprint(f\"  Unique training classes:    {len(name_groups)}\")\nprint(f\"  Merged (duplicates found):  {len(merged)}\")\nprint(f\"{'─'*50}\")\n\nif merged:\n    print(f\"\\n  Merged groups (REVIEW THESE):\")\n    for norm_key, orig_names in list(merged.items())[:10]:\n        print(f\"    Class: '{norm_key}'\")\n        for n in set(orig_names):\n            print(f\"      ← {n}\")\n        print()\nelse:\n    print(f\"\\n  No duplicates found — all names are unique.\")\n\nprint(f\"\\n  Sample classes (first 8):\")\nfor cid in list(class_map.keys())[:8]:\n    c = class_map[cid]\n    print(f\"    [{cid:>3s}] {c['name']:40s}  ({len(c['exemplar_dirs'])} source dirs)\")

---\n\n# STEP 3 — Build YOLO Dataset\n\nThis step takes the downloaded images and converts them into a YOLO-format training dataset:\n\n1. **Collect** all images (stills + extracted video frames) from `data/exemplars/`\n2. **Auto-label** catalog images using contour detection (instruments on neutral background)\n3. **Stratified split** into train / val / test — ensuring every class appears in all splits\n4. **Export** the final YOLO directory structure\n\n### What you'll get:\n```\ndata/yolo_dataset/\n  images/train/    images/val/    images/test/\n  labels/train/    labels/val/    labels/test/\n  data.yaml        ← ready for ultralytics\n```\n\n### Important:\n- Classes with < 3 images go to train only (can't split meaningfully)\n- The `class_distribution.json` file shows exactly how many images per class per split — **check this**

In [ ]:
## ── 3: Build YOLO dataset from exemplars ──\nimport shutil, random\n\nrandom.seed(42)\n\nTRAIN_RATIO = 0.70\nVAL_RATIO   = 0.15\n# TEST_RATIO = 1 - TRAIN_RATIO - VAL_RATIO = 0.15\n\nIMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}\n\n# ── 3A: Collect all images per class ──\nclass_images = {}  # class_id → [image_paths]\n\nfor cid, cinfo in class_map.items():\n    images = []\n    for dirname in cinfo['exemplar_dirs']:\n        d = DATA / \"exemplars\" / dirname\n        if d.exists():\n            images.extend([f for f in d.iterdir() if f.suffix.lower() in IMG_EXTS])\n    class_images[cid] = images\n\ntotal_imgs = sum(len(v) for v in class_images.values())\nclasses_with_data = sum(1 for v in class_images.values() if v)\nclasses_empty = sum(1 for v in class_images.values() if not v)\n\nprint(f\"{'─'*50}\")\nprint(f\"  STEP 3A: Image collection\")\nprint(f\"{'─'*50}\")\nprint(f\"  Total images found:          {total_imgs}\")\nprint(f\"  Classes with images:         {classes_with_data}\")\nprint(f\"  Classes with NO images:      {classes_empty}\")\nprint(f\"{'─'*50}\")\n\nif classes_empty:\n    print(f\"\\n  ⚠ Classes missing images (upload via SurgicalInstruments app):\")\n    for cid, cinfo in class_map.items():\n        if not class_images[cid]:\n            print(f\"    • {cinfo['name']}\")\n\nprint(f\"\\n  Image count per class (top 10 + bottom 5):\")\nsorted_classes = sorted(class_images.items(), key=lambda x: len(x[1]), reverse=True)\nfor cid, imgs in sorted_classes[:10]:\n    bar = '█' * min(len(imgs), 40)\n    print(f\"    [{cid:>3s}] {class_map[cid]['name']:35s} {len(imgs):4d} {bar}\")\nif len(sorted_classes) > 15:\n    print(f\"    ...\")\n    for cid, imgs in sorted_classes[-5:]:\n        bar = '█' * min(len(imgs), 40)\n        print(f\"    [{cid:>3s}] {class_map[cid]['name']:35s} {len(imgs):4d} {bar}\")

### 3B — Auto-label + Split + Export YOLO dataset\n\nFor catalog images (instrument on a neutral background), we generate bounding box labels automatically using contour detection. Then we split into train/val/test and create the YOLO directory structure.\n\nThe progress bar shows export progress. At the end you'll see the class distribution across splits.

In [ ]:
## ── 3B: Auto-label, split, export YOLO ──\nimport cv2\n\nYOLO_DIR = DATA / \"yolo_dataset\"\nfor split in ('train', 'val', 'test'):\n    (YOLO_DIR / \"images\" / split).mkdir(parents=True, exist_ok=True)\n    (YOLO_DIR / \"labels\" / split).mkdir(parents=True, exist_ok=True)\n\ndef auto_label_bbox(img_path):\n    \"\"\"Auto-generate YOLO bbox from contour detection (catalog images on neutral bg).\"\"\"\n    img = cv2.imread(str(img_path))\n    if img is None:\n        return None\n    h, w = img.shape[:2]\n    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)\n    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)\n    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n    if not contours:\n        return \"0.5 0.5 0.9 0.9\"  # fallback: full-frame bbox\n    largest = max(contours, key=cv2.contourArea)\n    x, y, bw, bh = cv2.boundingRect(largest)\n    # YOLO format: center_x center_y width height (normalized)\n    cx = (x + bw / 2) / w\n    cy = (y + bh / 2) / h\n    nw = bw / w\n    nh = bh / h\n    return f\"{cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}\"\n\nclass_dist = {}  # class_id → {train: N, val: N, test: N}\ntotal_exported = 0\n\nfor cid, imgs in class_images.items():\n    if not imgs:\n        continue\n    \n    random.shuffle(imgs)\n    n = len(imgs)\n    \n    if n < 3:\n        splits = [('train', imgs)]\n    else:\n        n_train = max(1, int(n * TRAIN_RATIO))\n        n_val = max(1, int(n * VAL_RATIO))\n        splits = [\n            ('train', imgs[:n_train]),\n            ('val',   imgs[n_train:n_train + n_val]),\n            ('test',  imgs[n_train + n_val:])\n        ]\n    \n    class_dist[cid] = {'train': 0, 'val': 0, 'test': 0}\n    \n    for split_name, split_imgs in splits:\n        for img_path in split_imgs:\n            # Copy image\n            dst_name = f\"{class_map[cid]['normalized'].replace(' ','_')}_{img_path.stem}{img_path.suffix}\"\n            shutil.copy2(img_path, YOLO_DIR / \"images\" / split_name / dst_name)\n            \n            # Generate label\n            bbox = auto_label_bbox(img_path)\n            label_path = YOLO_DIR / \"labels\" / split_name / f\"{Path(dst_name).stem}.txt\"\n            label_path.write_text(f\"{cid} {bbox}\\n\")\n            \n            class_dist[cid][split_name] += 1\n            total_exported += 1\n    \n    progress_bar(int(cid) + 1, len(class_map), prefix='Exporting YOLO')\n\n# Write data.yaml\nclass_names = {int(k): v['name'] for k, v in class_map.items()}\nyaml_content = f\"\"\"path: {YOLO_DIR.resolve()}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnc: {len(class_names)}\nnames: {json.dumps(class_names)}\n\"\"\"\n(YOLO_DIR / \"data.yaml\").write_text(yaml_content)\n\n# Save class distribution\nwith open(DATA / \"class_distribution.json\", \"w\") as f:\n    json.dump(class_dist, f, indent=2)\n\nprint(f\"\\n\\n{'─'*50}\")\nprint(f\"  STEP 3B RESULTS\")\nprint(f\"{'─'*50}\")\n\ntrain_total = sum(d['train'] for d in class_dist.values())\nval_total = sum(d['val'] for d in class_dist.values())\ntest_total = sum(d['test'] for d in class_dist.values())\n\nprint(f\"  Total exported:          {total_exported}\")\nprint(f\"  Train images:            {train_total}\")\nprint(f\"  Validation images:       {val_total}\")\nprint(f\"  Test images:             {test_total}\")\nprint(f\"  Classes in dataset:      {len(class_dist)}\")\nprint(f\"  data.yaml:               {YOLO_DIR / 'data.yaml'}\")\nprint(f\"{'─'*50}\")\n\nprint(f\"\\n  Split distribution per class:\")\nprint(f\"    {'Class':<35s} {'Train':>6s} {'Val':>6s} {'Test':>6s} {'Total':>6s}\")\nprint(f\"    {'─'*35} {'─'*6} {'─'*6} {'─'*6} {'─'*6}\")\nfor cid in sorted(class_dist.keys(), key=int):\n    d = class_dist[cid]\n    t = d['train'] + d['val'] + d['test']\n    name = class_map[cid]['name'][:35]\n    print(f\"    {name:<35s} {d['train']:>6d} {d['val']:>6d} {d['test']:>6d} {t:>6d}\")\nprint(f\"    {'─'*35} {'─'*6} {'─'*6} {'─'*6} {'─'*6}\")\nprint(f\"    {'TOTAL':<35s} {train_total:>6d} {val_total:>6d} {test_total:>6d} {total_exported:>6d}\")

---\n\n# STEP 4 — Visual Sanity Check\n\nBefore training, **always** spot-check the dataset. This cell displays a grid of sample images with their assigned labels. Look for:\n\n- **Wrong labels** — instrument name doesn't match the image\n- **Bad crops** — bounding box misses part of the instrument\n- **Duplicate images** — same image appearing multiple times\n- **Low quality** — blurry, dark, or unrecognizable images

In [ ]:
## ── 4: Visual sanity check — show sample images per class ──\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import Rectangle\n\nN_CLASSES_TO_SHOW = 6   # how many classes to display\nN_IMAGES_PER_CLASS = 3  # images per class\n\nfig, axes = plt.subplots(N_CLASSES_TO_SHOW, N_IMAGES_PER_CLASS, \n                         figsize=(4 * N_IMAGES_PER_CLASS, 3.5 * N_CLASSES_TO_SHOW))\n\n# Pick classes with the most images\nshow_classes = [cid for cid, imgs in sorted(class_images.items(), \n               key=lambda x: len(x[1]), reverse=True) if imgs][:N_CLASSES_TO_SHOW]\n\nfor row, cid in enumerate(show_classes):\n    imgs = class_images[cid]\n    samples = random.sample(imgs, min(N_IMAGES_PER_CLASS, len(imgs)))\n    for col in range(N_IMAGES_PER_CLASS):\n        ax = axes[row][col] if N_CLASSES_TO_SHOW > 1 else axes[col]\n        if col < len(samples):\n            img = cv2.imread(str(samples[col]))\n            if img is not None:\n                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\n                ax.imshow(img)\n                # Draw auto-label bbox\n                h, w = img.shape[:2]\n                bbox_str = auto_label_bbox(samples[col])\n                if bbox_str:\n                    parts = [float(x) for x in bbox_str.split()]\n                    cx, cy, bw, bh = parts\n                    rect = Rectangle(((cx - bw/2)*w, (cy - bh/2)*h), bw*w, bh*h,\n                                    linewidth=2, edgecolor='lime', facecolor='none')\n                    ax.add_patch(rect)\n        ax.set_xticks([])\n        ax.set_yticks([])\n        if col == 0:\n            ax.set_ylabel(class_map[cid]['name'][:25], fontsize=9, rotation=0, \n                         labelpad=120, va='center')\n\nplt.suptitle('Sample Images with Auto-Labels (green bbox)', fontsize=14, y=1.01)\nplt.tight_layout()\nplt.show()\nprint(\"  ↑ Review these samples. If labels look wrong, check the catalog data.\")

---\n\n# STEP 5 — Train YOLOv8-seg Detector\n\nThis calls the training script. It fine-tunes a YOLOv8-seg model on our instrument dataset.\n\n### What to expect:\n- **Duration:** 15-20 hours on RTX 4090 / A100 (50 epochs)\n- **Output:** Best weights saved to `runs/` folder\n- **Progress:** Ultralytics prints per-epoch metrics (mAP, loss, etc.)\n\n### Tuning knobs:\n| Parameter | Default | When to change |\n|---|---|---|\n| `epochs` | 50 | Increase if loss still dropping at epoch 50 |\n| `imgsz` | 1024 | Reduce to 640 if GPU runs out of memory |\n| `batch-size` | 4 | Reduce to 2 if GPU OOM; increase to 8 if you have 48GB+ VRAM |\n| `model` | yolov8m-seg | Use yolov8s-seg for faster training, yolov8l-seg for more accuracy |

In [ ]:
## ── 5: Train YOLOv8-seg ──\n## Adjust parameters below as needed:\n\nEPOCHS     = 50\nIMGSZ      = 1024\nBATCH_SIZE = 4\nMODEL      = \"yolov8m-seg.pt\"\nDATA_YAML  = str(YOLO_DIR / \"data.yaml\")\n\nimport torch\nprint(f\"{'─'*50}\")\nprint(f\"  PRE-FLIGHT CHECK\")\nprint(f\"{'─'*50}\")\nprint(f\"  CUDA available:    {torch.cuda.is_available()}\")\nif torch.cuda.is_available():\n    print(f\"  GPU:               {torch.cuda.get_device_name(0)}\")\n    print(f\"  VRAM:              {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB\")\nprint(f\"  Dataset:           {DATA_YAML}\")\nprint(f\"  Classes:           {len(class_map)}\")\nprint(f\"  Train images:      {train_total}\")\nprint(f\"  Model:             {MODEL}\")\nprint(f\"  Epochs:            {EPOCHS}\")\nprint(f\"  Image size:        {IMGSZ}\")\nprint(f\"  Batch size:        {BATCH_SIZE}\")\nprint(f\"{'─'*50}\")\nprint(f\"\\n  Ready to train. Run the next cell to start.\")\nprint(f\"  ⚠ This will take several hours on GPU.\")

In [ ]:
## ── 5B: START TRAINING (long-running) ──\n\nfrom ultralytics import YOLO\n\nmodel = YOLO(MODEL)\nresults = model.train(\n    data=DATA_YAML,\n    epochs=EPOCHS,\n    imgsz=IMGSZ,\n    batch=BATCH_SIZE,\n    device=0,\n    project=str(COMPONENT / \"runs\"),\n    name=\"instrument_detector\",\n    exist_ok=True,\n    verbose=True\n)\n\nprint(f\"\\n{'─'*50}\")\nprint(f\"  TRAINING COMPLETE\")\nprint(f\"{'─'*50}\")\nprint(f\"  Best weights: {COMPONENT / 'runs' / 'instrument_detector' / 'weights' / 'best.pt'}\")\nprint(f\"  Results dir:  {COMPONENT / 'runs' / 'instrument_detector'}\")\nprint(f\"{'─'*50}\")

---\n\n# STEP 6 — Evaluate Results\n\nAfter training, we evaluate the model on the test set.\n\n### Metrics explained:\n| Metric | What it measures | Target (Phase 1C) | Phase 1A estimate |\n|---|---|---|---|\n| **mAP@50** | Detection accuracy at 50% IoU overlap | >= 0.85 | 0.60-0.75 |\n| **mAP@50-95** | Detection accuracy averaged across IoU thresholds | >= 0.70 | 0.40-0.55 |\n| **Per-class F1** | Precision/recall balance per instrument | >= 0.80 | varies |\n\n### What to look for:\n- **Which classes perform worst?** → Need more training images for those (upload to catalog)\n- **Confusion matrix** → Which instruments get confused with each other?\n- **Per-class breakdown** → Are critical instruments (sponges, needles) above threshold?

In [ ]:
## ── 6: Evaluate on test set ──\n\nBEST_WEIGHTS = COMPONENT / \"runs\" / \"instrument_detector\" / \"weights\" / \"best.pt\"\n\nif not BEST_WEIGHTS.exists():\n    print(\"  ⚠ No trained weights found. Run Step 5 first.\")\nelse:\n    model = YOLO(str(BEST_WEIGHTS))\n    metrics = model.val(\n        data=DATA_YAML,\n        split='test',\n        imgsz=IMGSZ,\n        batch=BATCH_SIZE,\n        device=0,\n        verbose=False\n    )\n    \n    print(f\"{'─'*50}\")\n    print(f\"  STEP 6: EVALUATION RESULTS\")\n    print(f\"{'─'*50}\")\n    print(f\"  mAP@50:          {metrics.box.map50:.4f}\")\n    print(f\"  mAP@50-95:       {metrics.box.map:.4f}\")\n    print(f\"  Precision (avg):  {metrics.box.mp:.4f}\")\n    print(f\"  Recall (avg):     {metrics.box.mr:.4f}\")\n    print(f\"{'─'*50}\")\n    \n    # Per-class breakdown\n    print(f\"\\n  Per-class results:\")\n    print(f\"    {'Class':<35s} {'mAP@50':>8s} {'Prec':>8s} {'Recall':>8s} {'Status':>8s}\")\n    print(f\"    {'─'*35} {'─'*8} {'─'*8} {'─'*8} {'─'*8}\")\n    \n    if hasattr(metrics.box, 'ap50') and metrics.box.ap50 is not None:\n        for i, ap50 in enumerate(metrics.box.ap50):\n            name = class_names.get(i, f'class_{i}')[:35]\n            p = metrics.box.p[i] if hasattr(metrics.box, 'p') else 0\n            r = metrics.box.r[i] if hasattr(metrics.box, 'r') else 0\n            status = \"PASS\" if ap50 >= 0.75 else \"LOW\"\n            print(f\"    {name:<35s} {ap50:>8.3f} {p:>8.3f} {r:>8.3f} {status:>8s}\")\n    \n    print(f\"\\n  Full results saved to: {COMPONENT / 'runs' / 'instrument_detector'}\")

---\n\n# STEP 7 — Test on a Sample Image\n\nRun the trained detector on a single image to see what it produces. This is useful for quick visual verification before deploying to the production pipeline.

In [ ]:
## ── 7: Run inference on a sample image ──\n\nif not BEST_WEIGHTS.exists():\n    print(\"  ⚠ No trained weights found. Run Step 5 first.\")\nelse:\n    # Pick a random test image\n    test_images = list((YOLO_DIR / \"images\" / \"test\").glob(\"*\"))\n    if not test_images:\n        test_images = list((YOLO_DIR / \"images\" / \"val\").glob(\"*\"))\n    \n    if test_images:\n        sample = random.choice(test_images)\n        model = YOLO(str(BEST_WEIGHTS))\n        results = model.predict(str(sample), imgsz=IMGSZ, conf=0.25, verbose=False)\n        \n        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))\n        \n        # Original\n        img = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)\n        ax1.imshow(img)\n        ax1.set_title(f\"Original: {sample.name}\", fontsize=10)\n        ax1.axis('off')\n        \n        # Detection result\n        result_img = results[0].plot()\n        result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)\n        ax2.imshow(result_img)\n        ax2.set_title(\"Detections\", fontsize=10)\n        ax2.axis('off')\n        \n        plt.tight_layout()\n        plt.show()\n        \n        # Print detections\n        print(f\"\\n  Detections on {sample.name}:\")\n        for box in results[0].boxes:\n            cls_id = int(box.cls[0])\n            conf = float(box.conf[0])\n            name = class_names.get(cls_id, f'class_{cls_id}')\n            print(f\"    • {name:35s} confidence={conf:.2f}\")\n        if not results[0].boxes:\n            print(\"    No detections. Try lowering confidence threshold.\")\n    else:\n        print(\"  No test images found.\")

---\n\n# STEP 8 — Summary & Next Steps\n\nThis cell prints a final summary report: what was built, what the numbers look like, and what to do next.

In [ ]:
## ── 8: Summary report ──\n\nprint(\"=\"*60)\nprint(\"  INSTRUMENT TRACKER — PIPELINE SUMMARY\")\nprint(\"=\"*60)\nprint(f\"\")\nprint(f\"  DATA SOURCE\")\nprint(f\"  ───────────\")\nprint(f\"  Catalog app:         {API_URL}\")\nprint(f\"  Instruments synced:  {len(resp)}\")\nprint(f\"  Kit catalogs:        {len(kits)}\")\nprint(f\"  Families:            {len(families)}\")\nprint(f\"\")\nprint(f\"  DEDUPLICATION\")\nprint(f\"  ─────────────\")\nprint(f\"  Original names:      {len(resp)}\")\nprint(f\"  Unique classes:      {len(name_groups)}\")\nprint(f\"  Names merged:        {len(merged)} groups\")\nprint(f\"\")\nprint(f\"  DATASET\")\nprint(f\"  ───────\")\nprint(f\"  Total images:        {total_exported}\")\nprint(f\"  Train / Val / Test:  {train_total} / {val_total} / {test_total}\")\nprint(f\"  Classes:             {len(class_dist)}\")\nprint(f\"  YOLO data.yaml:      {YOLO_DIR / 'data.yaml'}\")\nprint(f\"\")\nprint(f\"  OUTPUT FILES\")\nprint(f\"  ────────────\")\nfor f in ['taxonomy.json', 'families.json', 'class_map.json', \n          'dedup_log.json', 'class_distribution.json']:\n    path = DATA / f\n    status = 'OK' if path.exists() else 'MISSING'\n    print(f\"  [{status:>7s}]  data/{f}\")\nprint(f\"  [{'OK' if (YOLO_DIR / 'data.yaml').exists() else 'MISSING':>7s}]  data/yolo_dataset/data.yaml\")\nweights = COMPONENT / \"runs\" / \"instrument_detector\" / \"weights\" / \"best.pt\"\nprint(f\"  [{'OK' if weights.exists() else 'NOT YET':>7s}]  runs/instrument_detector/weights/best.pt\")\nprint(f\"\")\nprint(f\"  NEXT STEPS\")\nprint(f\"  ──────────\")\nprint(f\"  1. Review dedup_log.json — are the right names merged?\")\nprint(f\"  2. Check class_distribution.json — any class too small?\")\nprint(f\"  3. Upload more images for underrepresented classes\")\nprint(f\"     via SurgicalInstruments app → re-run this notebook\")\nprint(f\"  4. When OR video recordings arrive (Sheba Phase 1A):\")\nprint(f\"     → Run generate_zero_shot.py for auto-annotation\")\nprint(f\"     → Human review in Label Studio\")\nprint(f\"     → Re-train with combined catalog + OR data\")\nprint(f\"\")\nprint(\"=\"*60)